# Conduction block: which distal anode configuration can stop the descending AP?

**Project:** FEM simulation of intraneural stimulation of the median nerve, block study
(walkthrough `docs/block.md`, decision D-12 in `docs/decisions.md`).
**Model:** patient UA01. Contacts 1–24 (stations −4…−1, the proximal site) are solved in 3D; the
distal contacts 25–48 are their mirror image in z (D-12).

**Where we are.** The first notebook (`notebooks/block_anode_cathode.ipynb`) followed the
supervisor's rule: the anode is the same contact as the cathode, on the other site (AS 2 → AS 44).
That anode **never blocks**. Through its virtual cathodes it fires the fibre from 17.5 µA
(≈ 1.7 × the stimulation current), with or without ramps, and no block was found up to 2 mA
(`docs/block.md` §6).

**What this notebook does.** As our own extension beyond the rule (to be shown to the tutor), it
keeps the proximal stimulation unchanged (AS 2 cathodic, aimed at fascicle F7) and compares **several
anode configurations on the distal site**, up to all 24 distal contacts together (a "wall"). For each
one it asks:
1. **Static screen** (seconds): how strong is the hyperpolarisation where we need block, compared
   with the strongest virtual cathode anywhere in the nerve?
2. **Map** (example fibre): over ramp × current, is there a window where the anode blocks without
   firing the fibre?
3. **Population check** (all 453 fibres) of the best setting: how many of the fibres excited by the
   proximal pulse are blocked, and how many fibres anywhere does the anode fire?
4. **Scoreboard**: one figure comparing all configurations.

**How to run**
1. Prerequisites as for the first notebook (`docs/block.md` §4): NEURON, `results/mrg/model/nrnmech.dll`.
2. The fibre profiles must include the distal contacts you want to test. After the 24-contact
   MATLAB run (`main_pipeline_3d` with `p.solve3d.sections = -4:-1`), run
   `matlab -batch "setup_paths; export_block_profiles"`. With the old 12-contact file the
   notebook still runs, but it **skips** every configuration that needs a contact not in the file
   (it prints which).
3. Do **not** run it while the MATLAB solve is running: both need a lot of RAM.
4. **Run All.** About 10–15 min with 7 worker processes (sections 7 and 9 take most of it).
   Outputs go to `results/block/anode_configs/`.

If a worker process fails, set `N_WORKERS = 1` (slower, everything in this process). After editing
`src/block/*.py`, restart the kernel.

**Every figure has a "How to read this figure" note** before the cell that draws it. The physics
(activating function, virtual cathodes, anodal block, anode break) is explained in section 1 of the
first notebook and summarised in section 1 below.

**Sections:** 1 background · 2 setup · 3 parameters · 4 data and anode configurations · 5 proximal
stimulation · 6 static screen · 7 maps · 8 one raster per configuration · 9 population check ·
10 scoreboard, summary, limitations.

## 1. Background in short: what a "wall" can and cannot do

- A fibre does not feel the **level** of the extracellular potential $V_e$, only its **curvature
  along the fibre**, i.e. the activating function $f_n = (V_{e,n-1} - 2V_{e,n} + V_{e,n+1})/\Delta x^2$
  at the nodes ($\Delta x$ = 1.15 mm for 10 µm fibres). $f > 0$ depolarises (towards firing),
  $f < 0$ hyperpolarises (towards block). A flat, uniformly high potential does nothing to the
  fibre. A "wall" works only through the **peak** of $V_e$ it creates along each fibre.
- Every peak of $V_e$ hyperpolarises the nodes at its centre and **depolarises the nodes on its
  flanks** (the virtual cathodes). The flanks cannot be removed, only made weaker relative to the
  centre.
- **Distance matters.** For a point anode at distance $r$ from the fibre, the ratio
  (centre hyperpolarisation) / (virtual-cathode peak) is about 2 when $r \ll \Delta x$ (the anode
  right next to the fibre, our intraneural contacts: 2.9 was measured) and grows to about 5 when
  $r \gtrsim \Delta x$ (this follows from the second derivative of $1/\sqrt{r^2 + z^2}$). A distant
  or spread-out anode gives relatively more block per unit of virtual-cathode risk. This is why block
  electrodes are usually cuffs. The cost is that a distant anode needs much more current.
- **A spread anode has a weak point.** If the configuration includes intraneural contacts, every
  fibre that passes close to one of them still feels mainly that one contact, with the near-contact
  ratio of ~2–3, but now with only a fraction of the total current. Those fibres can be fired by
  their local contact before the wall blocks anything.
- **Two ways the anode fires the fibre:** virtual-cathode excitation (at the onset, or during a
  ramp: in the first notebook the MRG fibre did not accommodate to 1 ms ramps) and anode break (at
  the offset, after a long hyperpolarisation).

**Success** for a setting means: on every fibre the proximal pulse excites, no AP reaches the muscle
and the brain gets the same APs as without the anode; and on every other fibre, the anode fires
nothing.

The fibre model is MRG (McIntyre, Richardson & Grill 2002) in NEURON: HH-type channels at the nodes
(fast and persistent Na, slow K, leak; mammalian, 37 °C) and an explicit myelin sheath (double cable),
driven by the FEM potential of each contact by superposition (`docs/block.md` §3,
`src/block/mrg_fibre.py`).

## 2. Setup

Same as the first notebook: imports, paths, NEURON check. `src/block/fibre_jobs.py` runs the fibre
simulations ("jobs") on several processes.

In [ ]:
import json, math, os, sys, time
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import scipy.io

# project root: Jupyter starts in notebooks/, the root is one level up
ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "block").exists():
    ROOT = ROOT.parent
assert (ROOT / "src" / "block" / "mrg_fibre.py").exists(), f"project root not found from {Path.cwd()}"

try:
    from neuron import h
except ImportError as err:
    raise ImportError("NEURON is not importable from this kernel (docs/block.md §4; start Jupyter from a "
                      "NEW terminal after installing NEURON).") from err
print("NEURON:", h.nrnversion())

sys.path.insert(0, str(ROOT / "src" / "block"))
from mrg_fibre import MRG_TABLE
import fibre_jobs

MECH_DIR = ROOT / "results" / "mrg" / "model"
BLOCK_DIR = ROOT / "results" / "block"
OUT_DIR = BLOCK_DIR / "anode_configs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("outputs:", OUT_DIR)

## 3. Parameters

Shared numbers (fibre model, proximal pulse, 10 % criterion, stimulation factor) come from
`config/block_params.m` through the JSON copy. The cell below holds the knobs of this exploration.

**Anode configurations.** Each entry is a list of distal contacts that work together as the anode.
The total anodal current is **split equally** among them: with 24 contacts at 1 mA total, each
carries 1 mA / 24 ≈ 42 µA. Two entries are special words, resolved in section 4:
- `"MIRROR"`: the contacts of the proximal pattern, mirrored (the supervisor's rule, as a reference);
- `"NEAREST"`: at each of the 4 distal stations, the shank contact closest to the centre of the
  target fascicle (a column of anodes along the nerve, aimed at the fascicle to block).

Distal contacts (`data/electrode/sites.mat`), z = +4.3 (station +1), +6.3 (+2), +8.3 (+3),
+10.3 mm (+4):

| station | shank contacts (r = 0.9, 0.3, 0.3, 0.9 mm) | shank angle | cuff pads | cuff angles |
|---|---|---|---|---|
| +1 | 25, 26, 27, 28 | 135° / 315° | 29, 30 | 225°, 45° |
| +2 | 31, 32, 33, 34 | 90° / 270° | 35, 36 | 180°, 0° |
| +3 | 37, 38, 39, 40 | 45° / 225° | 41, 42 | 135°, 315° |
| +4 | 43, 44, 45, 46 | 0° / 180° | 47, 48 | 90°, 270° |

Feel free to add your own configurations to the dictionary; the rest of the notebook adapts.

In [ ]:
PATIENT = "UA01"
P = json.loads((BLOCK_DIR / f"{PATIENT}_block_params.json").read_text())
MRG = P["mrg"]

# --- proximal stimulation (unchanged from the first notebook) ------------------------------------------
PROX_PATTERN = {2: -1.0}             # AS 2 cathodic (inside F7)
TARGET_FASCICLE = 7

# --- distal anode configurations: name -> list of contacts (equal split) or "MIRROR" / "NEAREST" ------
ANODE_CONFIGS = {
    "same contact (rule)":       "MIRROR",
    "station +4, all 6":         [43, 44, 45, 46, 47, 48],
    "station +4, 2 cuff pads":   [47, 48],
    "8 cuff pads (4 stations)":  [29, 30, 35, 36, 41, 42, 47, 48],
    "16 shank (4 stations)":     [25, 26, 27, 28, 31, 32, 33, 34, 37, 38, 39, 40, 43, 44, 45, 46],
    "all 24 distal (wall)":      list(range(25, 49)),
    "nearest to target, 4 st.":  "NEAREST",
}

FIBRE_DIAM_um = MRG["fibreDiam_um"]

# --- timing (anode times counted from the proximal onset) --------------------------------------------
T_PROX_ON_s  = 1.5e-3
PROX_WIDTH_s = P["pulse"]["stimWidth_s"]
T_AFTER_s    = P["pulse"]["tAfter_s"]
PLATEAU_START_s  = 0.1e-3            # the plateau must contain the AP arrival at EVERY anode station
PLATEAU_END_s    = 0.9e-3            # (~0.35 ms at station +1 ... ~0.49 ms at station +4; checked in 5)
RISE_TIMES_s     = [0.0, 0.1e-3, 0.3e-3, 0.6e-3, 1.0e-3]
FALL_EQUALS_RISE = True
RAMP_STEP_s      = 10e-6

# total anodal current of the configuration, log-spaced; wider than in the first notebook because a
# spread or distant anode needs more current
ANODE_AMPS_A = np.geomspace(10e-6, 5e-3, 22)                 # 10 uA ... 5 mA (each ~1.35 x the previous)

N_CHECK_PER_CONFIG = 1               # best settings per configuration re-checked on all fibres
N_WORKERS = max(1, (os.cpu_count() or 2) - 1)

assert T_PROX_ON_s + PLATEAU_START_s - max(RISE_TIMES_s) >= 0.2e-3, "increase T_PROX_ON_s"
assert MRG["nodeAlignment"] == "proximalStation", "one shared fibre object needs nodeAlignment = proximalStation"
print(f"{len(ANODE_CONFIGS)} configurations, {len(RISE_TIMES_s)} ramps, {len(ANODE_AMPS_A)} currents; "
      f"{N_WORKERS} worker processes")

## 4. Data and anode configurations

The profile file holds, for every fibre and every available contact, $V(z)$ for 1 A on that contact
(see the first notebook, section 4). Here we also read the whole electrode layout, turn the
configurations into patterns `{contact: weight}`, and skip those whose contacts are not in the file.

In [ ]:
H5_PATH = BLOCK_DIR / f"{PATIENT}_fibre_profiles.h5"
with h5py.File(H5_PATH, "r") as f:
    z_m           = f["z_m"][:].ravel()
    fibre_x_m     = f["x_m"][:].ravel()
    fibre_y_m     = f["y_m"][:].ravel()
    fascicle      = f["fascicle"][:].ravel().astype(int)
    contact_id    = f["contact_id"][:].ravel().astype(int)
    mirrored_from = f["mirrored_from"][:].ravel().astype(int)
    contact_z_m   = f["contact_z_m"][:].ravel()
    n_contacts, n_fibres, n_z = f["V_perA"].shape
    fem_source = f.attrs["source"]
fem_source = fem_source.decode() if isinstance(fem_source, bytes) else str(fem_source)
AVAILABLE = set(contact_id.tolist())
CONTACT_Z = dict(zip(contact_id.tolist(), contact_z_m.tolist()))
print(f"FEM solution: {fem_source}")
print(f"{n_fibres} fibres, {fascicle.max()} fascicles; {n_contacts} contacts in the file: {sorted(AVAILABLE)}")

# whole electrode layout (48 rows), lengths in mm, angles in degrees
S = scipy.io.loadmat(str(ROOT / "data" / "electrode" / "sites.mat"))["sites"]
SITES = pd.DataFrame(S, columns=["AS", "type", "section", "z_mm", "r_mm", "theta_deg", "phi_deg",
                                 "thickness_mm", "sideZ_mm", "sideXY_mm"])
SITES["AS"] = SITES["AS"].astype(int)
SITES = SITES.set_index("AS")
SITES["kind"] = np.where(SITES["type"] == 1, "shank", "cuff")
CONTACT_AREA_m2 = float(SITES["sideZ_mm"].iloc[0] * SITES["sideXY_mm"].iloc[0]) * 1e-6


def layout_partners(sites_all):
    """Proximal contact -> its partner on the other site, from the nominal layout (sites.mat): same
    type, same radius and angle, opposite station (section -k -> +k). The pads of the Gmsh model are
    moved by the random lead insertion, so this is the "same electrode" of the device, not a geometric
    mirror. (With the old PDE Toolbox model and 24 solved contacts this equals the mirror partners.)"""
    out = {}
    for c, s in sites_all[sites_all.section < 0].iterrows():
        cand = sites_all[(sites_all.section == -s.section) & (sites_all.type == s.type) &
                         (np.isclose(sites_all.theta_deg, s.theta_deg)) &
                         ((sites_all.r_mm.isna() & np.isnan(s.r_mm)) | np.isclose(sites_all.r_mm, s.r_mm))]
        if len(cand) == 1:
            out[int(c)] = int(cand.index[0])
    return out

MIRROR = layout_partners(SITES)                               # proximal -> distal "same electrode"

# target fascicle centre (from its fibres), used by "NEAREST"
tx, ty = fibre_x_m[fascicle == TARGET_FASCICLE].mean() * 1e3, fibre_y_m[fascicle == TARGET_FASCICLE].mean() * 1e3


def nearest_shank_per_station(x_mm, y_mm):
    """At each distal station, the shank contact closest to the point (x_mm, y_mm)."""
    ids = []
    for sec in (1, 2, 3, 4):
        sh = SITES[(SITES.section == sec) & (SITES.type == 1)]
        d = np.hypot(sh.r_mm * np.cos(np.radians(sh.theta_deg)) - x_mm, sh.r_mm * np.sin(np.radians(sh.theta_deg)) - y_mm)
        ids.append(int(d.idxmin()))
    return ids


ANODE_PATTERNS, rows = {}, []
for name, spec in ANODE_CONFIGS.items():
    if spec == "MIRROR":
        ids = [MIRROR[c] for c in PROX_PATTERN if c in MIRROR]
    elif spec == "NEAREST":
        ids = nearest_shank_per_station(tx, ty)
    else:
        ids = [int(c) for c in spec]
    missing = sorted(set(ids) - AVAILABLE) if ids else ["(no partner)"]
    if missing:
        rows.append({"configuration": name, "contacts": ids, "status": f"SKIPPED: not in the file {missing}"})
        continue
    assert all(SITES.loc[c, "section"] > 0 for c in ids), f"{name}: anode contacts must be on the distal site"
    ANODE_PATTERNS[name] = {c: 1.0 / len(ids) for c in sorted(ids)}           # equal split, anodic (> 0)
    zz = [CONTACT_Z[c] * 1e3 for c in ids]
    rows.append({"configuration": name, "contacts": sorted(ids), "n": len(ids),
                 "shank": int(sum(SITES.loc[c, "type"] == 1 for c in ids)),
                 "cuff": int(sum(SITES.loc[c, "type"] == 2 for c in ids)),
                 "z range [mm]": f"{min(zz):+.1f} ... {max(zz):+.1f}", "status": "ok"})
CONFIG_TABLE = pd.DataFrame(rows)
assert ANODE_PATTERNS, "no configuration can be run with this profile file"
CONFIG_TABLE

### How to read this figure: the distal contacts on the cross-section

The slice of the nerve as in the first notebook: each grey dot is one fibre (the target fascicle F7
in colour). On top, **every distal contact**, coloured by station (+1 … +4). The stations are at
different z, so in reality they are 2 mm apart along the nerve; the figure flattens them onto one
slice to show their angles.
- **Squares** = shank (intraneural) contacts. The shank turns by 45° from one station to the next,
  so the 16 shank contacts point in 4 directions.
- **Thick bars** = cuff pads on the nerve surface. `sites.mat` does not give their radius, so they
  are drawn on an approximate outline just outside the fibres.
- The number next to each marker is the contact id used in `ANODE_CONFIGS`.

Use it to see which contacts are close to the target fascicle (good for local block, but also close
to its fibres, so they risk the virtual cathodes) and which are far.

In [ ]:
a_mm = np.abs(fibre_x_m).max() * 1e3 + 0.25            # approximate outline for the cuff pads
b_mm = np.abs(fibre_y_m).max() * 1e3 + 0.25
st_col = {1: "tab:blue", 2: "tab:orange", 3: "tab:green", 4: "tab:red"}
fig, ax = plt.subplots(figsize=(7.5, 6.2))
other = fascicle != TARGET_FASCICLE
ax.scatter(fibre_x_m[other] * 1e3, fibre_y_m[other] * 1e3, s=5, color="0.75")
ax.scatter(fibre_x_m[~other] * 1e3, fibre_y_m[~other] * 1e3, s=8, color="tab:purple", label=f"F{TARGET_FASCICLE} (target)")
for k in range(1, fascicle.max() + 1):
    sel = fascicle == k
    ax.text(fibre_x_m[sel].mean() * 1e3, fibre_y_m[sel].mean() * 1e3, f"F{k}", fontsize=8, ha="center", va="center")
tt = np.linspace(0, 2 * np.pi, 200)
ax.plot(a_mm * np.cos(tt), b_mm * np.sin(tt), color="0.6", lw=0.6, ls=":")
for c, s in SITES[SITES.section > 0].iterrows():
    th = np.radians(s.theta_deg)
    col = st_col[int(s.section)]
    if s.type == 1:
        x, y = s.r_mm * np.cos(th), s.r_mm * np.sin(th)
        ax.plot(x, y, "s", ms=7, mfc="none", mec=col, mew=1.5)
    else:
        x, y = a_mm * np.cos(th), b_mm * np.sin(th)
        ax.plot(x, y, "_", ms=14, mew=4, color=col)
    ax.annotate(str(c), (x, y), xytext=(4, 4), textcoords="offset points", fontsize=7, color=col)
for sec, col in st_col.items():
    ax.plot([], [], "s", mfc="none", mec=col, label=f"station +{sec} (z = {SITES[SITES.section == sec].z_mm.iloc[0]:+.1f} mm)")
ax.set_aspect("equal")
ax.set(xlabel="x [mm]", ylabel="y [mm]", title="Distal contacts (squares = shank, bars = cuff pads, approx.)")
ax.legend(fontsize=7, loc="upper right")
fig.savefig(OUT_DIR / "distal_contacts.png", dpi=130, bbox_inches="tight")
plt.show()

### How to read this figure: which contacts each configuration uses

A table drawn as an image. **Each row is one configuration**, **each column one distal contact**
(25 → 48, grouped by station and separated by white lines; S = shank, C = cuff). A coloured cell
means the contact belongs to that configuration as an anode. The shade is the fraction of the total
current it carries: dark for a single contact (100 %), light when the current is shared among many.

In [ ]:
# Which distal contacts each anode configuration uses (S = shank / intraneural, C = cuff), one row per
# configuration, one column block per station.
dist_ids = sorted(SITES[SITES.section > 0].index)
names = list(ANODE_PATTERNS)
M = np.array([[ANODE_PATTERNS[n].get(c, 0.0) for c in dist_ids] for n in names])
fig, ax = plt.subplots(figsize=(12, 0.9 + 0.45 * len(names)))
im = ax.imshow(np.where(M > 0, M, np.nan), cmap="Reds", vmin=0, vmax=1, aspect="auto")
ax.set_yticks(range(len(names)), names)
ax.set_xticks(range(len(dist_ids)), [f"{c}\n{'S' if SITES.loc[c, 'type'] == 1 else 'C'}" for c in dist_ids], fontsize=7)
for k in range(1, 4):
    ax.axvline(6 * k - 0.5, color="w", lw=3)
for k, sec in enumerate((1, 2, 3, 4)):
    ax.text(6 * k + 2.5, -0.9, f"station +{sec}", ha="center", fontsize=8)
ax.set_xticks(np.arange(-0.5, len(dist_ids)), minor=True)
ax.set_yticks(np.arange(-0.5, len(names)), minor=True)
ax.grid(which="minor", color="0.85", lw=0.5)
fig.colorbar(im, ax=ax, label="fraction of the anodal current")
ax.set_title("Anode configurations (rows) and the distal contacts they use (columns)", pad=18, fontsize=10)
fig.savefig(OUT_DIR / "configurations.png", dpi=130, bbox_inches="tight")
plt.show()

## 5. Proximal stimulation, example fibre, timing

Same as the first notebook: build the fibre, compute the threshold of the target fascicle (10 % of
its fibres) and I_stim = 1.2 × that, find which fibres the proximal pulse excites, choose the example
fibre (the most excitable fibre of the target fascicle), and measure when its AP reaches each distal
station. The anode plateau must cover the arrival at **every** station a configuration uses.

In [ ]:
# The shared MRG fibre (node under the proximal contact) and the pulse helpers: rectangle, trapezoid (ramps
# as a staircase of short pulses), proximal drive, anode pulses, end of a run, charge density on the most
# loaded contact.
Z_PROX_m = CONTACT_Z[next(iter(PROX_PATTERN))]
t0 = time.time()
runner = fibre_jobs.FibreRunner(H5_PATH, MRG, FIBRE_DIAM_um, Z_PROX_m, MECH_DIR, N_WORKERS)
FIBRE = runner.fibre
print(f"MRG fibre: {FIBRE.n_nodes} nodes, internode {FIBRE.internode_m*1e3:.2f} mm; started in {time.time()-t0:.1f} s")


def rect(t_on_s, width_s, amp_A):
    """One rectangular pulse (t_on, t_off, amplitude)."""
    return [(t_on_s, t_on_s + width_s, amp_A)]


def trapezoid(t_plateau_on_s, t_plateau_off_s, amp_A, rise_s, fall_s, step_s=RAMP_STEP_s):
    """Trapezoid as a staircase: the rise ENDS at t_plateau_on_s, the fall STARTS at t_plateau_off_s."""
    pulses = []
    n = int(round(rise_s / step_s))
    if n > 0:
        e = t_plateau_on_s - rise_s + np.arange(n + 1) * rise_s / n
        pulses += [(e[k], e[k + 1], amp_A * (k + 0.5) / n) for k in range(n)]
    pulses.append((t_plateau_on_s, t_plateau_off_s, amp_A))
    n = int(round(fall_s / step_s))
    if n > 0:
        e = t_plateau_off_s + np.arange(n + 1) * fall_s / n
        pulses += [(e[k], e[k + 1], amp_A * (1 - (k + 0.5) / n)) for k in range(n)]
    return pulses


def charge_C(pulses):
    return sum((off - on) * a for on, off, a in pulses)


def end_s(pulses):
    return max(off for _, off, _ in pulses)


def prox_drive(amp_A):
    return (PROX_PATTERN, rect(T_PROX_ON_s, PROX_WIDTH_s, amp_A))


def anode_pulses(amp_A, rise_s):
    fall_s = rise_s if FALL_EQUALS_RISE else 0.0
    return trapezoid(T_PROX_ON_s + PLATEAU_START_s, T_PROX_ON_s + PLATEAU_END_s, amp_A, rise_s, fall_s)


def t_stop_for(*pulse_lists):
    return max(end_s(p) for p in pulse_lists) + T_AFTER_s


def max_density_uC_cm2(pattern, pulses):
    """Charge density per phase on the most loaded contact [uC/cm^2]."""
    return charge_C(pulses) * max(pattern.values()) / (CONTACT_AREA_m2 * 1e4) * 1e6


T_STOP_PROX_s = T_PROX_ON_s + PROX_WIDTH_s + T_AFTER_s

In [ ]:
# threshold of the target fascicle (10 % of its fibres) and stimulation current
t0 = time.time()
tgt = np.flatnonzero(fascicle == TARGET_FASCICLE)
jobs = [(int(i), PROX_PATTERN, T_PROX_ON_s, PROX_WIDTH_s, T_STOP_PROX_s,
         P["searchStart_A"], P["searchMax_A"], P["searchTol"]) for i in tgt]
th_prox_A = np.full(n_fibres, np.nan)
th_prox_A[tgt] = runner.map(fibre_jobs.threshold_job, jobs)
n_need = max(1, math.ceil(P["recruitFraction"] * len(tgt)))
finite = np.sort(th_prox_A[tgt][np.isfinite(th_prox_A[tgt])])
TH10_A = finite[n_need - 1]
I_STIM = P["stimFactor"] * TH10_A
print(f"F{TARGET_FASCICLE}: 10%-of-fibres threshold {TH10_A*1e6:.2f} uA; I_stim = {I_STIM*1e6:.2f} uA  [{time.time()-t0:.0f} s]")

# which fibres the proximal pulse excites (reference outcome of every fibre)
res = runner.run([(int(i), [prox_drive(I_STIM)], T_STOP_PROX_s) for i in range(n_fibres)])
EXPECT_BRAIN = np.array([r[0] for r in res])
EXCITED = np.array([sum(r) > 0 for r in res])
print(f"the proximal pulse excites {EXCITED.sum()} fibres, in fascicles {sorted(set(fascicle[EXCITED].tolist()))}  "
      f"[{time.time()-t0:.0f} s]")

# example fibre and arrival times at every distal station
cand = np.flatnonzero(EXCITED & (fascicle == TARGET_FASCICLE))
I_EX = int(cand[np.nanargmin(th_prox_A[cand])])
_, _, rec = runner.record(I_EX, [prox_drive(I_STIM)], T_STOP_PROX_s)


def first_crossing_s(rec, node):
    v = rec["v_V"][node]
    up = (v[:-1] < MRG["apThreshold_V"]) & (v[1:] >= MRG["apThreshold_V"])
    return rec["t_s"][1:][up][0] if up.any() else np.nan


ARRIVAL_s = {sec: first_crossing_s(rec, FIBRE.node_index(SITES[SITES.section == sec].z_mm.iloc[0] * 1e-3)) - T_PROX_ON_s
             for sec in (1, 2, 3, 4)}
print(f"example fibre {I_EX} (F{fascicle[I_EX]}); AP arrival after the proximal onset: " +
      ", ".join(f"station +{k} {v*1e3:.2f} ms" for k, v in ARRIVAL_s.items()))
used = {int(SITES.loc[c, "section"]) for p in ANODE_PATTERNS.values() for c in p}
late = [k for k in used if not (PLATEAU_START_s < ARRIVAL_s[k] < PLATEAU_END_s)]
print("plateau covers all arrivals" if not late else f"WARNING: plateau misses the arrival at stations {late}")

## 6. Static screen: hyperpolarisation where we need block vs virtual cathodes anywhere

Before simulating the dynamics, a cheap comparison with the activating function (no NEURON). For
each configuration and 1 µA of total anodal current, it computes $f_n$ at the nodes of **every**
fibre, in the zone of the anode (from 3.5 mm before its first station to 3.5 mm after its last).

### How to read this figure: activating function of each configuration (example fibre)

One small panel per configuration, all with the same horizontal axis: position along the nerve in the
distal region (mm). Each dot is one node of the example fibre. Above zero = depolarised (towards
firing), below zero = hyperpolarised (towards block). The dotted vertical lines mark the four distal
stations.
- **Look at the shape.** A good blocking configuration has a deep negative region where the AP must
  pass, with small positive flanks.
- **Look at the scale of each panel** (each has its own vertical axis). Spread configurations have
  much smaller values per µA, so they need more current. What matters is the *ratio* negative /
  positive, which is in the table below.

In [ ]:
zn = FIBRE.z_node_m
dz = z_m[1] - z_m[0]
i0 = np.clip(np.floor((zn - z_m[0]) / dz).astype(int), 0, len(z_m) - 2)
fr = (zn - z_m[i0]) / dz


def af_all_fibres(pattern, amp_A=1e-6):
    """Activating function [V/m^2] at the nodes of every fibre, shape (nFibres, nNodes); NaN at the end nodes."""
    Vp = fibre_jobs.pattern_profiles_all(pattern) * amp_A          # (nFibres, nZ)
    ve = Vp[:, i0] * (1 - fr) + Vp[:, i0 + 1] * fr                  # interpolated at the nodes
    af = np.full_like(ve, np.nan)
    af[:, 1:-1] = (ve[:, :-2] - 2 * ve[:, 1:-1] + ve[:, 2:]) / FIBRE.internode_m ** 2
    return af


def anode_window(pattern, margin_m=3.5e-3):
    zz = [CONTACT_Z[c] for c in pattern]
    return (zn >= min(zz) - margin_m) & (zn <= max(zz) + margin_m)


AF = {name: af_all_fibres(pat) for name, pat in ANODE_PATTERNS.items()}
station_z = [SITES[SITES.section == s].z_mm.iloc[0] for s in (1, 2, 3, 4)]
fig, axs = plt.subplots(len(AF), 1, figsize=(10, 1.7 * len(AF)), sharex=True, squeeze=False)
for ax, (name, af) in zip(axs[:, 0], AF.items()):
    ax.plot(zn * 1e3, af[I_EX], ".-", color="tab:red")
    ax.axhline(0, color="grey", lw=0.5)
    for zs in station_z:
        ax.axvline(zs, color="grey", ls=":", lw=0.8)
    ax.set_ylabel("V/m² per µA", fontsize=7)
    ax.set_title(name, fontsize=8.5, loc="left")
axs[-1, 0].set_xlim(0, 15)
axs[-1, 0].set_xlabel("z along the nerve [mm]  (dotted = distal stations +1 ... +4)")
fig.suptitle(f"Activating function at the nodes of fibre {I_EX} (F{fascicle[I_EX]}) for 1 uA of total anodal current",
             fontsize=10)
fig.tight_layout()
fig.savefig(OUT_DIR / "af_per_configuration.png", dpi=130, bbox_inches="tight")
plt.show()

### How to read the screen table

One row per configuration; all values per µA of total anodal current, in the anode zone.
- `hyp on excited fibres`: the **weakest** hyperpolarisation among the fibres the proximal pulse
  excites. These are the fibres we must block, and block is limited by the least hyperpolarised one.
- `max virtual cathode (any fibre)`: the **strongest** depolarisation anywhere in the nerve, and the
  fascicle of that fibre. That fibre is the first one the anode will fire.
- **`global ratio`** = hyp on excited fibres / max virtual cathode. **The higher, the better.** It
  is a static proxy for "how much block we get before the anode fires something somewhere". The
  same-contact anode gives the baseline. A configuration that does not improve on it is unlikely to
  work.
- `ratio on example fibre`: the local ratio on the example fibre alone, as in the first notebook.
- `rough fire current`: the current at which the strongest virtual cathode equals the depolarisation
  that fires the example fibre under the proximal cathode (50 µs). Only an order of magnitude:
  the long anodal pulse fires at a lower current.

The static screen ignores time (ramps, the moment the AP arrives) and the nonlinear membrane, so the
simulations of sections 7–9 have the last word.

In [ ]:
af_cath = af_all_fibres(PROX_PATTERN)[I_EX]
near_prox = np.abs(zn - Z_PROX_m) < 3.5e-3
cath_peak = np.nanmax(af_cath[near_prox])                     # depolarisation that fires the example fibre at th
rows = []
for name, af in AF.items():
    w = anode_window(ANODE_PATTERNS[name])
    hyp = -np.nanmin(af[:, w], axis=1)                       # hyperpolarisation of each fibre (positive number)
    vc = np.nanmax(af[:, w], axis=1)                         # virtual-cathode peak of each fibre
    hyp_exc = hyp[EXCITED].min()
    i_vc = int(np.nanargmax(vc))
    rows.append({"configuration": name,
                 "hyp on excited fibres": hyp_exc,
                 "max virtual cathode (any fibre)": vc[i_vc],
                 "fascicle of that fibre": f"F{fascicle[i_vc]}",
                 "global ratio": hyp_exc / vc[i_vc],
                 "ratio on example fibre": hyp[I_EX] / vc[I_EX] if vc[I_EX] > 0 else np.inf,
                 "rough fire current [uA]": th_prox_A[I_EX] * 1e6 * cath_peak / vc[i_vc]})
SCREEN = pd.DataFrame(rows)
SCREEN.to_csv(OUT_DIR / "static_screen.csv", index=False)
SCREEN.round(3)

## 7. Maps: ramp × current for every configuration (example fibre)

For every configuration, ramp and current, two runs of the example fibre: **anode alone** (does it
fire the fibre?) and **proximal + anode** (is the descending AP stopped?). Outcomes:

| outcome | meaning | colour | letter |
|---|---|---|---|
| anode fires | the anode alone sends an AP somewhere | red | F |
| blocked | anode alone silent; with the proximal pulse no AP to the muscle and the brain gets its AP | green | B |
| passes | anode alone silent, but the AP still reaches the muscle (too weak) | grey | P |
| other | anything else | yellow | O |

The example fibre is the fibre the proximal pulse excites most easily (in F7, next to AS 2). Being
right next to AS 44, it is the hardest case for configurations that use AS 44, and an easier one for
the others. Section 9 checks all fibres.

In [ ]:
CATEGORIES = ["passes", "blocked", "anode fires", "other"]
CAT_COLOURS = ["#9e9e9e", "#2e7d32", "#c62828", "#f9a825"]
CAT_LETTERS = ["P", "B", "F", "O"]


def categorise(alone, combined, expect_brain):
    if sum(alone) > 0:
        return "anode fires"
    if combined[1] == 0 and combined[0] == expect_brain:
        return "blocked"
    if combined[1] > 0:
        return "passes"
    return "other"


def setting_jobs(i_fibre, pattern, pulses):
    t_stop = t_stop_for(rect(T_PROX_ON_s, PROX_WIDTH_s, I_STIM), pulses)
    return [(int(i_fibre), [(pattern, pulses)], t_stop),
            (int(i_fibre), [prox_drive(I_STIM), (pattern, pulses)], t_stop)]


grid = [(n, r, a) for n in ANODE_PATTERNS for r in RISE_TIMES_s for a in ANODE_AMPS_A]
jobs = []
for n, r, a in grid:
    jobs += setting_jobs(I_EX, ANODE_PATTERNS[n], anode_pulses(a, r))
print(f"{len(grid)} settings x 2 = {len(jobs)} simulations on {N_WORKERS} processes ...")
t0 = time.time()
out = runner.run(jobs)
print(f"done in {time.time()-t0:.0f} s")

rows = []
for k, (n, r, a) in enumerate(grid):
    alone, comb = out[2 * k], out[2 * k + 1]
    pulses = anode_pulses(a, r)
    rows.append({"configuration": n, "rise_s": r, "amp_A": a, "rise_ms": r * 1e3, "amp_uA": a * 1e6,
                 "alone_brain": alone[0], "alone_muscle": alone[1], "comb_brain": comb[0], "comb_muscle": comb[1],
                 "outcome": categorise(alone, comb, EXPECT_BRAIN[I_EX]),
                 "charge_nC": charge_C(pulses) * 1e9,
                 "max_density_uC_cm2": max_density_uC_cm2(ANODE_PATTERNS[n], pulses)})
MAP = pd.DataFrame(rows)
MAP.to_csv(OUT_DIR / "maps_example_fibre.csv", index=False)
pd.crosstab(MAP.configuration, MAP.outcome).reindex(list(ANODE_PATTERNS))

### How to read this figure: the outcome maps

One chessboard per configuration (same layout as in the first notebook):
- **column = total anodal current** of the configuration (left weak, right strong, log scale);
- **row = ramp length** (bottom = rectangular pulse);
- **colour and letter = outcome** (table above).

Reading one row left to right you expect grey (too weak), then green or red. **Green cells are the
goal**: currents where the configuration blocks without firing. Compare the boards from top to
bottom: does a spread or distant configuration turn red cells green, and at what current? Remember
the current is the **total** current: with many contacts each one carries only a share.

In [ ]:
# Outcome maps: for every configuration, rise time (rows) x anodal current (columns), the outcome of the
# example fibre (blocked / passes / anode fires / ...).
names = list(ANODE_PATTERNS)
fig, axs = plt.subplots(len(names), 1, figsize=(12.5, 1.0 + 1.25 * len(names)), sharex=True, squeeze=False)
for ax, n in zip(axs[:, 0], names):
    g = MAP[MAP.configuration == n]
    M = np.array([[CATEGORIES.index(g[(g.rise_s == r) & (g.amp_A == a)].outcome.iloc[0]) for a in ANODE_AMPS_A]
                  for r in RISE_TIMES_s])
    ax.imshow(M, cmap=ListedColormap(CAT_COLOURS), vmin=-0.5, vmax=3.5, aspect="auto", origin="lower")
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, CAT_LETTERS[M[i, j]], ha="center", va="center", fontsize=6, color="w")
    ax.set_yticks(range(len(RISE_TIMES_s)), [f"{r*1e3:.1f}" for r in RISE_TIMES_s], fontsize=7)
    ax.set_ylabel("ramp [ms]", fontsize=7)
    ax.set_title(n, fontsize=9, loc="left")
axs[-1, 0].set_xticks(range(len(ANODE_AMPS_A)), [f"{a*1e6:.0f}" for a in ANODE_AMPS_A], rotation=60, fontsize=7)
axs[-1, 0].set_xlabel(f"total anodal current [uA]   (I_stim = {I_STIM*1e6:.1f} uA)")
fig.legend(handles=[Patch(color=c, label=f"{l} ({s})") for c, l, s in zip(CAT_COLOURS, CATEGORIES, CAT_LETTERS)],
           loc="upper right", ncol=4, fontsize=8)
fig.suptitle(f"Outcome on the example fibre {I_EX} (F{fascicle[I_EX]})", x=0.25, fontsize=10)
fig.tight_layout(rect=(0, 0, 1, 0.97))
fig.savefig(OUT_DIR / "maps_example_fibre.png", dpi=130, bbox_inches="tight")
plt.show()

**Windows in numbers.** One row per configuration and ramp: the lowest current at which the anode
fires the fibre, the range of currents that block, and the charge and charge density per phase on
the most loaded contact at the lowest blocking current. A useful row has `first_block_uA` below
`first_fire_uA`. The charge density is reported but not compared with the safety limits (Shannon,
charge-injection capacity of gold): that comparison is still to do.

In [ ]:
# Table per configuration and rise time: first current that fires, first and last current that blocks,
# number of blocking settings and the charge at the first block.
rows = []
for (n, r), g in MAP.groupby(["configuration", "rise_s"], sort=False):
    blk, fire = g[g.outcome == "blocked"], g[g.outcome == "anode fires"]
    first = blk.iloc[0] if len(blk) else None
    rows.append({"configuration": n, "rise_ms": r * 1e3,
                 "first_fire_uA": fire.amp_uA.min() if len(fire) else np.nan,
                 "first_block_uA": blk.amp_uA.min() if len(blk) else np.nan,
                 "last_block_uA": blk.amp_uA.max() if len(blk) else np.nan,
                 "n_blocking": len(blk),
                 "charge_at_first_block_nC": first.charge_nC if first is not None else np.nan,
                 "density_at_first_block_uC_cm2": first.max_density_uC_cm2 if first is not None else np.nan})
WIN = pd.DataFrame(rows)
WIN.to_csv(OUT_DIR / "windows_example_fibre.csv", index=False)
WIN.round(1)

## 8. One raster per configuration

For each configuration, one representative run of the example fibre with the proximal pulse:
- if the configuration blocks somewhere: its **lowest-charge blocking setting** (should show the
  upward stripe stopping at the anode);
- otherwise: the **lowest current at which the anode fires**, with the longest ramp (shows where and
  when the anode fires the fibre).

### How to read this figure: the rasters

Each panel is a space–time map of the fibre (full explanation in the first notebook, section 6):
**x = time** from the proximal onset, **y = position** along the fibre (brain at the bottom, muscle
at the top), **colour = membrane voltage** (dark purple = rest, bright = AP, black = hyperpolarised).
- An AP is a bright tilted stripe; the ">" from the proximal line (cyan, −10.3 mm) is the stimulus.
- The dashed lines in the distal region mark the stations used by the configuration. While the anode
  is on, the rows there turn black (hyperpolarised) and the rows beside them lighter (virtual
  cathodes).
- **Success looks like:** the stripe going down reaches the bottom, the stripe going up **stops**
  at the anode zone, and no new ">" starts there. The title gives the counts at the two ends.

In [ ]:
# Raster helper and one raster of the example fibre per configuration, at a representative setting (its
# lowest-charge block; if it never blocks, its first firing with the longest ramp).
def plot_raster(ax, rec, title, pattern):
    t, v = rec["t_s"], rec["v_V"]
    im = ax.pcolormesh((t - T_PROX_ON_s) * 1e3, zn * 1e3, v * 1e3, cmap="magma", vmin=-100, vmax=40, shading="auto")
    ax.axhline(Z_PROX_m * 1e3, color="c", lw=0.8, ls="--")
    for zz in sorted({CONTACT_Z[c] for c in pattern}):
        ax.axhline(zz * 1e3, color="w", lw=0.6, ls="--")
    ax.set_title(title, fontsize=7.5)
    return im


REP = {}
for n in names:
    g = MAP[MAP.configuration == n]
    blk = g[g.outcome == "blocked"]
    if len(blk):
        row = blk.loc[blk.charge_nC.idxmin()]
        REP[n] = (row.rise_s, row.amp_A, "lowest-charge block")
    else:
        fire = g[(g.outcome == "anode fires") & (g.rise_s == max(RISE_TIMES_s))]
        row = fire.iloc[0] if len(fire) else g.iloc[-1]
        REP[n] = (row.rise_s, row.amp_A, "first firing, longest ramp" if len(fire) else "strongest tested")

ncol = 3
nrow = math.ceil(len(names) / ncol)
fig, axs = plt.subplots(nrow, ncol, figsize=(5 * ncol, 4.2 * nrow), sharex=True, sharey=True, squeeze=False)
for ax, n in zip(axs.ravel(), names):
    r, a, why = REP[n]
    pulses = anode_pulses(a, r)
    nb, nm, rec = runner.record(I_EX, [prox_drive(I_STIM), (ANODE_PATTERNS[n], pulses)],
                                t_stop_for(rect(T_PROX_ON_s, PROX_WIDTH_s, I_STIM), pulses))
    im = plot_raster(ax, rec, f"{n}: {a*1e6:.0f} uA, ramp {r*1e3:.1f} ms ({why})\n-> brain {nb}, muscle {nm}",
                     ANODE_PATTERNS[n])
for ax in axs.ravel()[len(names):]:
    ax.axis("off")
for ax in axs[-1]:
    ax.set_xlabel("time from the proximal onset [ms]")
for ax in axs[:, 0]:
    ax.set_ylabel("z [mm] (brain bottom, muscle top)")
fig.colorbar(im, ax=axs, label="membrane potential [mV]", shrink=0.6)
fig.savefig(OUT_DIR / "rasters_per_configuration.png", dpi=120, bbox_inches="tight")
plt.show()

## 9. Population check: the best setting of each configuration on all fibres

For each configuration that blocks the example fibre, its best setting (per ramp: the blocking
current in the geometric middle of the blocking range; then the ramp with the lowest charge) is
simulated on all fibres:
- **anode alone** on all 453 fibres: which fibres does the anode fire? (off-target);
- **proximal + anode** on the fibres the proximal pulse excites: are they blocked?

### How to read this figure: outcome on the cross-section

One nerve slice per checked configuration, one dot per fibre, coloured by what happened:
**green** = excited by the proximal pulse and blocked (wanted); **blue** = excited and the AP still
reached the muscle; **red** = fired by the anode (unwanted); **light grey** = quiet (correct for
fibres the proximal pulse does not excite). A good configuration shows only green and grey; red dots
show where the anode spills over.

In [ ]:
settings = []
for n in names:
    cands = []
    for r, g in MAP[MAP.configuration == n].groupby("rise_s", sort=False):
        blk = g[g.outcome == "blocked"]
        if len(blk):
            centre = np.exp(np.log(blk.amp_A).mean())
            a = float(blk.amp_A.iloc[np.argmin(np.abs(np.log(blk.amp_A) - np.log(centre)))])
            cands.append((charge_C(anode_pulses(a, r)), float(r), a))
    settings += [(n, r, a) for _, r, a in sorted(cands)[:N_CHECK_PER_CONFIG]]
print(f"{len(settings)} settings to check on all fibres" if settings else "no configuration blocks the example fibre")

exc = np.flatnonzero(EXCITED)
POP_ROWS, FIBRE_ROWS = [], []
for n, r, a in settings:
    t0 = time.time()
    pat, pulses = ANODE_PATTERNS[n], anode_pulses(a, r)
    t_stop = t_stop_for(rect(T_PROX_ON_s, PROX_WIDTH_s, I_STIM), pulses)
    out = runner.run([(int(i), [(pat, pulses)], t_stop) for i in range(n_fibres)] +
                     [(int(i), [prox_drive(I_STIM), (pat, pulses)], t_stop) for i in exc])
    alone, comb = out[:n_fibres], dict(zip(exc.tolist(), out[n_fibres:]))
    fires = np.array([sum(x) > 0 for x in alone])
    status = np.where(fires, "anode fires", "quiet")
    for i in exc:
        if not fires[i]:
            status[i] = categorise(alone[i], comb[i], EXPECT_BRAIN[i])
    POP_ROWS.append({"configuration": n, "rise_ms": r * 1e3, "amp_uA": a * 1e6,
                     "charge_nC": charge_C(pulses) * 1e9, "max_density_uC_cm2": max_density_uC_cm2(pat, pulses),
                     "excited_by_proximal": len(exc), "blocked": int(np.sum(status[exc] == "blocked")),
                     "anode_fires_total": int(fires.sum()),
                     "anode_fires_other_fascicles": int(np.sum(fires & (fascicle != TARGET_FASCICLE))),
                     "fascicles_fired": sorted(set(fascicle[fires].tolist()))})
    FIBRE_ROWS += [{"configuration": n, "fibre": i, "fascicle": fascicle[i], "status": status[i]} for i in range(n_fibres)]
    print(f"   {n}: {a*1e6:.0f} uA, ramp {r*1e3:.1f} ms -> blocked {POP_ROWS[-1]['blocked']}/{len(exc)}, "
          f"anode fires {fires.sum()} fibres  [{time.time()-t0:.0f} s]")
POP = pd.DataFrame(POP_ROWS)
POP.to_csv(OUT_DIR / "population_summary.csv", index=False)
pd.DataFrame(FIBRE_ROWS).to_csv(OUT_DIR / "population_per_fibre.csv", index=False)

if POP_ROWS:
    FR = pd.DataFrame(FIBRE_ROWS)
    colours = {"blocked": "#2e7d32", "passes": "#5c6bc0", "other": "#f9a825", "anode fires": "#c62828", "quiet": "#d0d0d0"}
    ncol = min(3, len(settings))
    nrow = math.ceil(len(settings) / ncol)
    fig, axs = plt.subplots(nrow, ncol, figsize=(5 * ncol, 4.4 * nrow), squeeze=False)
    for ax, (n, r, a) in zip(axs.ravel(), settings):
        g = FR[FR.configuration == n].set_index("fibre")
        ax.scatter(fibre_x_m * 1e3, fibre_y_m * 1e3, s=9, c=[colours[g.status[i]] for i in range(n_fibres)])
        ax.set_aspect("equal")
        ax.set_title(f"{n}\n{a*1e6:.0f} uA, ramp {r*1e3:.1f} ms", fontsize=8.5)
    for ax in axs.ravel()[len(settings):]:
        ax.axis("off")
    fig.legend(handles=[Patch(color=c, label=l) for l, c in colours.items()], loc="lower center", ncol=5, fontsize=8)
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(OUT_DIR / "population_cross_sections.png", dpi=130, bbox_inches="tight")
    plt.show()
POP

## 10. Scoreboard, summary, limitations

### How to read this figure: the scoreboard

One row per configuration, three panels side by side.
1. **Window on the example fibre** (log current axis). The red ✕ is the lowest current at which the
   anode fires the fibre; the green ● is the lowest current that blocks (over all ramps). **A window
   exists when the green dot is to the left of the red cross.** "no block" means no green cell in
   its map.
2. **Fibres blocked**: of the fibres the proximal pulse excites, how many are blocked at the checked
   setting (section 9). The full bar means all of them.
3. **Fibres fired by the anode** at that setting, anywhere in the nerve (off-target). Zero is the
   goal.

A configuration that works has a window in panel 1, a full bar in panel 2, and zero in panel 3.

In [ ]:
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(14, 0.8 + 0.55 * len(names)), sharey=True,
                                 gridspec_kw={"width_ratios": [2, 1, 1]})
y = np.arange(len(names))
for k, n in enumerate(names):
    w = WIN[WIN.configuration == n]
    ff, fb = w.first_fire_uA.min(), w.first_block_uA.min()
    if np.isfinite(ff):
        a1.plot(ff, k, "x", color="#c62828", ms=9, mew=2)
    if np.isfinite(fb):
        a1.plot(fb, k, "o", color="#2e7d32", ms=8)
    else:
        a1.text(ANODE_AMPS_A[-1] * 1e6, k, "no block ", ha="right", va="center", fontsize=8, color="0.3")
    p = POP[POP.configuration == n] if len(POP) else POP
    if len(p):
        a2.barh(k, p.blocked.iloc[0], color="#2e7d32")
        a3.barh(k, p.anode_fires_total.iloc[0], color="#c62828")
        a3.text(p.anode_fires_total.iloc[0], k, f" {p.anode_fires_total.iloc[0]}", va="center", fontsize=8)
    else:
        a2.text(0.1, k, "not checked", va="center", fontsize=8, color="0.4")
a1.set_xscale("log")
a1.set_xlim(ANODE_AMPS_A[0] * 1e6 * 0.8, ANODE_AMPS_A[-1] * 1e6 * 1.2)
a1.set(yticks=y, yticklabels=names, xlabel="total anodal current [uA]",
       title="example fibre: x = anode fires, o = blocks")
a1.invert_yaxis()
a2.set(xlim=(0, max(1, int(EXCITED.sum()))), xlabel=f"blocked (of {int(EXCITED.sum())} excited)",
       title="population: blocked")
a3.set(xlabel="fibres fired by the anode (of 453)", title="population: off-target")
for ax in (a1, a2, a3):
    ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "scoreboard.png", dpi=130, bbox_inches="tight")
plt.show()

**Limitations** (same as the first notebook, plus two specific to spread anodes):
- Multi-contact anodes are built by superposition of single-contact solutions in which the idle
  contacts are insulating (D-07). With many active contacts this is an approximation: in reality the
  idle metal contacts redistribute the current.
- The 24 distal contacts are the mirror image of contacts 1–24 (valid while the nerve section is the
  same along z, D-12).
- The current is split equally among the contacts of a configuration; other weightings (e.g. more
  current on the cuff pads) could do better.
- One fibre diameter (10 µm), straight fibres, node aligned on the proximal station, quasi-static
  tissue, MRG without strong accommodation, **no safety check** (charge densities reported only).

In [ ]:
# Summary of the run written to summary.json.
summary = {
    "patient": PATIENT, "fem_source": fem_source, "fibre_diam_um": FIBRE_DIAM_um,
    "prox_pattern": PROX_PATTERN, "target_fascicle": TARGET_FASCICLE,
    "th10_target_uA": TH10_A * 1e6, "I_stim_uA": I_STIM * 1e6, "excited_by_proximal": int(EXCITED.sum()),
    "example_fibre": I_EX, "arrival_ms": {f"+{k}": v * 1e3 for k, v in ARRIVAL_s.items()},
    "anode_patterns": ANODE_PATTERNS, "skipped": CONFIG_TABLE[CONFIG_TABLE.status != "ok"].configuration.tolist(),
    "static_screen": SCREEN.to_dict(orient="records"),
    "windows_example_fibre": WIN.to_dict(orient="records"),
    "population": POP.to_dict(orient="records") if len(POP_ROWS) else [],
}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=1,
                                                 default=lambda o: o.item() if hasattr(o, "item") else str(o)))
runner.close()
print("wrote", OUT_DIR / "summary.json", "and stopped the worker processes")